# Sesión 05 - Lab 1: Limpieza y estandarización (Bronze → Silver)

Este laboratorio trabaja con un extracto de un CRM interno (`clientes_crm.csv`) que llega con los problemas de calidad típicos de una fuente sin ningún control de antemano: nulos en llaves obligatorias y en datos opcionales, mayúsculas y minúsculas inconsistentes, y clientes reingestados más de una vez con datos actualizados. A diferencia del `clientes_lab1` de la Sesión 04 (datos reales de AdventureWorksLT, ya bastante limpios), este dataset se armó a propósito para que cada paso de limpieza tenga un efecto visible. El flujo sigue el Diagrama 1 del canvas: aterrizar el archivo en Bronze, detectar nulos, decidir entre `fillna`/`dropna`, estandarizar tipos y formatos, deduplicar por llave, y validar el esquema final antes de escribir. Tanto Bronze (`dbassociate.bronze.clientes_crm`) como Silver (`dbassociate.silver.clientes_crm`) usan los schemas reales de Medallion, creados desde el instructivo de la Sesión 01.

## Verificación del entorno

In [0]:
# dbutils.fs.ls("/Volumes/dbassociate/default/vol_landing/sesion_05")
dbutils.fs.ls("abfss://metastore-data@saassociatedbkrs.dfs.core.windows.net/sesion5")

## Lab 1-Bronze: Aterrizar el extracto del CRM en Bronze

El archivo crudo aterriza tal como llegó del CRM, sin cambiarle nada, más las mismas columnas de auditoría usadas desde la Sesión 01 (`ingestion_timestamp`, `source_system`, `batch_id`).

En criollo: `cliente_id` se define como texto (`StringType`), no como número, aunque en casi todas las filas sea un número. La razón es simple: algunas filas llegan sin `cliente_id`, y si Spark tuviera que forzar esa columna a número, ese vacío rompería la lectura. Por eso todo se guarda tal cual llegó, como texto, y recién en Silver (después de decidir qué hacer con los nulos) se convierte cada columna a su tipo real.

In [0]:
from datetime import datetime
from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql.types import StructType, StructField, StringType

schema_crm = StructType([
    StructField("cliente_id", StringType(), True),
    StructField("nombre", StringType(), True),
    StructField("apellido", StringType(), True),
    StructField("email", StringType(), True),
    StructField("telefono", StringType(), True),
    StructField("empresa", StringType(), True),
    StructField("segmento", StringType(), True),
    StructField("asesor_comercial", StringType(), True),
    StructField("fecha_actualizacion", StringType(), True),
])

df_crudo = spark.read.csv(
    # "/Volumes/dbassociate/default/vol_landing/sesion_05/clientes_crm.csv",
    "abfss://metastore-data@saassociatedbkrs.dfs.core.windows.net/sesion5/clientes_crm.csv",
    header=True,
    schema=schema_crm,
)

df_bronze_nuevo = (
    df_crudo
    .withColumn("ingestion_timestamp", current_timestamp()) ## para hacerlo standar el UTC debe ser 0
    .withColumn("source_system", lit("crm_clientes"))
    .withColumn("batch_id", lit("carga_" + datetime.now().strftime("%Y%m%d_%H%M")))
)

# pasamos los datos a bronze
spark.sql("CREATE SCHEMA IF NOT EXISTS dbassociate.bronze")

df_bronze_nuevo.write.mode("overwrite").saveAsTable("dbassociate.bronze.clientes_crm")


print("Filas aterrizadas en Bronze:", df_bronze_nuevo.count())

In [0]:
%sql
select * from dbassociate.bronze.clientes_crm

## Lab 1A: Explorar Bronze y detectar nulos

Antes de decidir cómo limpiar, hay que saber qué tan sucia está la tabla: cuántas filas tiene cada columna con valor nulo. Es como revisar un formulario antes de corregirlo: primero contás cuántos campos vinieron en blanco, después decidís qué hacer con cada uno.

In [0]:
from pyspark.sql.functions import col, count, when

df_bronze = spark.table("dbassociate.bronze.clientes_crm")

# cantidad de nulos por columna
resumen_nulos = df_bronze.select([
    count(when(col(c).isNull(), c)).alias(c) for c in df_bronze.columns
])

print("Filas totales en Bronze:", df_bronze.count())
resumen_nulos.show(truncate=False)

## Lab 1B: Limpieza de nulos

Hay dos formas de tratar un dato faltante, según qué tan importante sea esa columna. Si falta algo sin lo cual el registro no sirve para nada (`cliente_id` o `email`: sin uno no sabés de quién es la fila, sin el otro no podés contactarlo), esa fila se descarta con `dropna`. Si falta algo útil pero no imprescindible (`telefono` o `empresa`: el cliente sigue siendo válido igual), se completa con un valor por defecto usando `fillna`, para no perder el resto de la información de esa fila.

En este extracto del CRM esperá ver descartadas varias filas reales por llave obligatoria nula, a diferencia de `clientes_lab1` en la Sesión 04, donde ese caso prácticamente no aparecía.

In [0]:
columnas_obligatorias = ["cliente_id", "email"] # para dropna
valores_por_defecto = { # para fillna
    "telefono": "Sin telefono",
    "empresa": "Sin empresa",
}

filas_antes = df_bronze.count()
# Eliminamos los registros con nulos en las columnas obligatorias
df_sin_nulos_obligatorios = df_bronze.dropna(subset=columnas_obligatorias)

filas_descartadas = filas_antes - df_sin_nulos_obligatorios.count()
print("Filas descartadas por llave obligatoria nula:", filas_descartadas)

# Asignamos valores por defecto a las columnas que puedan tener nulos
df_limpio = df_sin_nulos_obligatorios.fillna(valores_por_defecto)

## Lab 1C: Estandarización de tipos y formatos

Para la computadora, `ANA@MAIL.COM` y `ana@mail.com` son dos textos distintos, aunque para nosotros sean el mismo correo. Por eso `email` se normaliza a minúsculas y sin espacios antes de usarla para comparar o agrupar: en este dataset hay clientes con el correo en mayúsculas completas, y sin esta normalización dos filas del mismo cliente podrían leerse como valores distintos. `nombre`/`apellido` se recortan y capitalizan por la misma razón de consistencia (algunos llegaron en mayúsculas, otros en minúsculas).

In [0]:
from pyspark.sql.functions import trim, lower, initcap

df_estandarizado = (
    df_limpio
    .withColumn("email", trim(lower(col("email")))) # minuscula
    .withColumn("nombre", trim(initcap(col("nombre")))) # capitalizar, es decir, la primera letra en mayuscula
    .withColumn("apellido", trim(initcap(col("apellido"))))
    .withColumn("empresa", trim(col("empresa")))
)

df_estandarizado.select("cliente_id", "nombre", "apellido", "email", "empresa").show(40, truncate=False)

## Lab 1D: Deduplicar por llave con una función de ventana

Bronze quedó **append-only**: algunos clientes (`4012`, `4020`, `4033`) llegaron dos veces, simulando una reingesta con datos actualizados. Antes de escribir Silver hay que quedarse con una sola fila por cliente.

Lo que hace `Window.partitionBy("cliente_id").orderBy(...)` junto con `row_number()` es: agrupar todas las filas por `cliente_id` (como armar un montoncito de papeles por cliente), y dentro de cada montoncito numerarlas del 1 en adelante según el orden que le pidamos (acá, la más reciente primero). Después nos quedamos solo con la fila número 1 de cada montoncito, la más actualizada. A diferencia de la Sesión 04 contra AdventureWorksLT (donde esta misma técnica no eliminaba ninguna fila, porque no había duplicados reales), acá sí se ve una reducción real en el conteo.

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import row_number

ventana_cliente = Window.partitionBy("cliente_id").orderBy(col("fecha_actualizacion").desc())

df_deduplicado = (
    df_estandarizado
    .withColumn("version_mas_reciente", row_number().over(ventana_cliente)) # asignamos un contador
    .filter(col("version_mas_reciente") == 1) # tomamos el primero por cliente
    .drop("version_mas_reciente")
)

# deduplicar indica quitar los datos repetidos o duplicados
print("Filas antes de deduplicar:", df_estandarizado.count())
print("Filas despues de deduplicar:", df_deduplicado.count())

## Lab 1E: Validar esquema y escribir en Silver

Silver define su propio esquema explícito, no se hereda el de Bronze sin revisión. En criollo, `cast()` le dice a Spark "tratá esta columna como si fuera de este otro tipo": `cliente_id` pasa de texto a número entero, y `fecha_actualizacion` pasa de texto a fecha real (no un texto que dice fecha). Son las conversiones que Bronze deja pendientes a propósito. La tabla se escribe en `dbassociate.silver`, no en `default` ni en `bronze`: es la primera tabla de este curso que vive en un schema real de Medallion, de punta a punta.

In [0]:
from pyspark.sql.types import IntegerType, StringType, DateType

# Asignamos el tipo de dato.
df_silver = df_deduplicado.select(
    col("cliente_id").cast(IntegerType()),
    col("nombre").cast(StringType()),
    col("apellido").cast(StringType()),
    col("email").cast(StringType()),
    col("telefono").cast(StringType()),
    col("empresa").cast(StringType()),
    col("segmento").cast(StringType()),
    col("asesor_comercial").cast(StringType()),
    col("fecha_actualizacion").cast(DateType()),
)

df_silver.printSchema()

# Para este ejemplo en bronze definimos el tipado, pero lo normal es que silver ya defina uno
spark.sql("CREATE SCHEMA IF NOT EXISTS dbassociate.silver")

df_silver.write.mode("overwrite").saveAsTable("dbassociate.silver.clientes_crm")

print("Filas escritas en Silver:", df_silver.count())

## Consulta de validación

In [0]:
spark.sql("""
    SELECT COUNT(*) AS total_silver, COUNT(DISTINCT cliente_id) AS clientes_unicos,
           SUM(CASE WHEN email IS NULL THEN 1 ELSE 0 END) AS emails_nulos
    FROM dbassociate.silver.clientes_crm
""").show()

## Limpieza

In [0]:
spark.sql("DROP TABLE IF EXISTS dbassociate.bronze.clientes_crm")
spark.sql("DROP TABLE IF EXISTS dbassociate.silver.clientes_crm")

print("Tablas temporales de este laboratorio eliminadas.")